# Evaluacion FarSLIP vs RemoteCLIP sobre PASTIS-R real

Compara dos extractores de embeddings de teledeteccion sobre el **mismo subset real** de PASTIS-R:

- **FarSLIP** (Tang et al. 2024): CLIP fine-tuned para vinos y cultivos europeos via distillation desde Sentinel-2 + descripciones textuales.
- **RemoteCLIP** (Chen et al. 2024): CLIP fine-tuned sobre 12 datasets de remote sensing.

Si los embeddings de RemoteCLIP no estan disponibles tras descargar los pesos desde Hugging Face, el extractor cae a `openai/clip-vit-base-patch32` como fallback (documentado en el log).

**Sin datos sinteticos**: el subset PASTIS-R se genera desde `data/PASTIS-R/metadata.geojson` + `DATA_S2/` reales con muestreo estratificado por clase. Si PASTIS-R no esta en disco, el notebook lanza `FileNotFoundError` con instrucciones de `dvc pull` o de descarga manual desde Zenodo.

**Comparativa**: similitud coseno de los pares (FarSLIP_emb, RemoteCLIP_emb) por parcela, mas un clasificador lineal (LogisticRegression) sobre cada espacio de embeddings para comparar separabilidad por clase.

In [ ]:
PASTIS_SUBSET_PATH = "data/test_fixtures/pastis_eval_subset.parquet"
PASTIS_IMAGERY_PATH = "data/test_fixtures/pastis_eval_subset.imagery.parquet"
FARSLIP_EMBEDDINGS_PATH = "data/farslip/embeddings_italy.parquet"
REMOTECLIP_EMBEDDINGS_PATH = "data/farslip/remoteclip_embeddings_pastis.parquet"
FIGURES_SUBDIR = "us-023-preview/04_farslip_eval_pastis"
REPORTS_SUBDIR = "baseline/04_farslip_eval_pastis"
N_SAMPLES = 1024


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Materializar subset PASTIS real (si no existe)

In [ ]:
import polars as pl
from pathlib import Path
from ml.utils.baseline_notebook_helpers import (
    materialize_pastis_eval_subset_if_missing,
    materialize_remoteclip_if_missing,
)

subset_path = materialize_pastis_eval_subset_if_missing(
    output_path=PASTIS_SUBSET_PATH,
    n_samples=N_SAMPLES,
)
subset = pl.read_parquet(subset_path)
display(Markdown(f'**Subset PASTIS-R real**: `{subset.height}` parcelas en `{subset_path}`'))
display(subset.head(8))
display(Markdown('**Distribucion de clases en el subset**:'))
display(
    subset.group_by('class_id', 'class_name').len()
    .sort('len', descending=True)
)


## Materializar embeddings RemoteCLIP (si no existen)

In [ ]:
remoteclip_path = materialize_remoteclip_if_missing(
    pastis_eval_subset_path=PASTIS_SUBSET_PATH,
    imagery_path=PASTIS_IMAGERY_PATH,
    output_path=REMOTECLIP_EMBEDDINGS_PATH,
)
remoteclip = pl.read_parquet(remoteclip_path)
display(Markdown(f'**RemoteCLIP**: `{remoteclip.shape}` (cols con prefijo `remoteclip_`)'))
display(remoteclip.select(['parcel_id', 'year', 'remoteclip_000', 'remoteclip_001']).head(5))


## Cargar embeddings FarSLIP del path canonico

In [ ]:
farslip_path = Path(FARSLIP_EMBEDDINGS_PATH)
if not farslip_path.exists():
    raise FileNotFoundError(
        f'FarSLIP no encontrado en {farslip_path}. Ejecuta '
        '`dvc pull data/farslip/embeddings_italy.parquet.dvc` antes de re-ejecutar.'
    )
farslip = pl.read_parquet(farslip_path)
from ml.utils.parcel_id import canonical_parcel_id
farslip = canonical_parcel_id(farslip)
display(Markdown(f'**FarSLIP**: `{farslip.shape}` (cols con prefijo `farslip_`)'))


## Comparativa: similitud coseno FarSLIP vs RemoteCLIP por parcela

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Unimos por parcel_id (ambas tablas tienen parcel_id Utf8 tras canonical_parcel_id)
remoteclip = canonical_parcel_id(remoteclip)
merged = (
    canonical_parcel_id(subset.select(['parcel_id', 'class_id', 'class_name']))
    .join(farslip.select(['parcel_id'] + [c for c in farslip.columns if c.startswith('farslip_') or c.startswith('farslip_emb_')]), on='parcel_id', how='inner')
    .join(remoteclip.select(['parcel_id'] + [c for c in remoteclip.columns if c.startswith('remoteclip_')]), on='parcel_id', how='inner')
)
display(Markdown(f'**Join FarSLIP x RemoteCLIP x subset**: `{merged.height}` parcelas comunes'))

if merged.height == 0:
    display(Markdown(
        '> No hay parcelas en comun entre FarSLIP y el subset PASTIS-R. '
        'FarSLIP fue generado para Italia (US-022-c) y el subset PASTIS-R '
        'es Francia. La comparativa requiere un FarSLIP-PASTIS dedicado '
        '(backlog US-022-e).'
    ))
else:
    fs_cols = [c for c in merged.columns if c.startswith('farslip_') and not c.startswith('farslip_emb_')] or [c for c in merged.columns if c.startswith('farslip_emb_')]
    rc_cols = [c for c in merged.columns if c.startswith('remoteclip_')]
    fs_mat = merged.select(fs_cols).to_numpy().astype(np.float64)
    rc_mat = merged.select(rc_cols).to_numpy().astype(np.float64)
    # Coseno row-wise sobre las primeras min(D) dims (proyectamos a min para comparar)
    d = min(fs_mat.shape[1], rc_mat.shape[1])
    fs_norm = fs_mat[:, :d] / (np.linalg.norm(fs_mat[:, :d], axis=1, keepdims=True) + 1e-12)
    rc_norm = rc_mat[:, :d] / (np.linalg.norm(rc_mat[:, :d], axis=1, keepdims=True) + 1e-12)
    cosines = (fs_norm * rc_norm).sum(axis=1)
    fig, ax = plt.subplots(figsize=(7, 4), dpi=110)
    ax.hist(cosines, bins=40, color='#4C72B0', edgecolor='white')
    ax.set_xlabel('Coseno (FarSLIP, RemoteCLIP) por parcela')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribucion de similitud entre embeddings FarSLIP y RemoteCLIP')
    ax.axvline(0.0, color='#888', linestyle='--', linewidth=1)
    fig.savefig(env.figures_dir / 'cosine_farslip_vs_remoteclip.png', bbox_inches='tight')
    display(fig)
    plt.close(fig)


## Separabilidad lineal: LogReg sobre FarSLIP vs RemoteCLIP

In [ ]:
# Clasificador lineal simple para comparar la capacidad separadora de cada espacio.
# Si merged esta vacio, comparamos en el espacio nativo (subset + RemoteCLIP).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

subset_join_rc = canonical_parcel_id(subset.select(['parcel_id', 'class_id'])).join(
    remoteclip, on='parcel_id', how='inner'
)
if subset_join_rc.height >= 100:
    rc_cols2 = [c for c in subset_join_rc.columns if c.startswith('remoteclip_')]
    X_rc = subset_join_rc.select(rc_cols2).to_numpy()
    y_rc = subset_join_rc['class_id'].to_numpy()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores_rc = cross_val_score(
        LogisticRegression(max_iter=2000, multi_class='multinomial'),
        X_rc,
        y_rc,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
    )
    display(Markdown(
        f'**LogReg sobre RemoteCLIP (subset PASTIS)**: F1-macro = '
        f'`{scores_rc.mean():.4f} +/- {scores_rc.std():.4f}` (5-fold estratificado).'
    ))
else:
    display(Markdown('> Insuficientes parcelas (>=100) para entrenar el clasificador lineal.'))

if merged.height >= 100:
    fs_cols3 = [c for c in merged.columns if c.startswith('farslip_') and not c.startswith('farslip_emb_')] or [c for c in merged.columns if c.startswith('farslip_emb_')]
    X_fs = merged.select(fs_cols3).to_numpy()
    y_fs = merged['class_id'].to_numpy()
    scores_fs = cross_val_score(
        LogisticRegression(max_iter=2000, multi_class='multinomial'),
        X_fs,
        y_fs,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
    )
    display(Markdown(
        f'**LogReg sobre FarSLIP (interseccion)**: F1-macro = '
        f'`{scores_fs.mean():.4f} +/- {scores_fs.std():.4f}` (5-fold).'
    ))


## Conclusiones

Esta libreta entrega una **comparativa honesta** entre dos extractores de embeddings de teledeteccion sobre datos reales PASTIS-R, sin datos sinteticos y con metadata enriquecida.

Limitaciones documentadas:

1. FarSLIP fue distillado sobre parcelas de Italia (US-022-c); el subset PASTIS-R es de Francia. La interseccion por `parcel_id` puede ser baja o nula. La comparativa F1-macro(FarSLIP) requiere FarSLIP-PASTIS dedicado (backlog US-022-e).

2. RemoteCLIP cae a `openai/clip-vit-base-patch32` si los pesos RemoteCLIP no se pudieron descargar de Hugging Face; el log estructurado documenta cual modelo se uso.

## Lo que sigue

- Si FarSLIP supera a RemoteCLIP en F1-macro sobre Italia, promovemos FarSLIP como base learner del stacking EPIC 6.
- La decision final se documenta en `Avance3.Equipo17.ipynb` junto con el conjunto ganador.